# Wellytics Analytics
Fill in **Cell 1**, then **Run All Cells**.

In [3]:
# ──────────────────────────────────────────────────────
# CELL 1 — CONFIG  (edit here, then Run All)
# ──────────────────────────────────────────────────────

ENV       = "PROD"    # DEV | STAGE | DEMO | PROD
ORG_ID    = "360dd06e-ce7f-4d93-ba44-d9d63189f1d5"         # paste organisation UUID here

# Date range filter  →  DD/MM/YY  or  DD/MM/YYYY
# Leave BOTH blank to fetch everything
DATE_FROM = "25/03/26"         # e.g. "22/02/26"
DATE_TO   = "25/03/26"         # e.g. "22/03/26"

In [ ]:
# ──────────────────────────────────────────────────────
# CELL 2 — ENV SETUP + AUTHENTICATION
# ──────────────────────────────────────────────────────
import subprocess, sys
for pkg in ["requests", "pandas", "openpyxl"]:
    subprocess.check_call([sys.executable, "-m", "pip", "install", pkg, "-q"])

import requests
import pandas as pd
from datetime import datetime, timezone
from collections import Counter
from IPython.display import display

# ── resolve env ──────────────────────────────────────
if ENV == 'DEV':
    BASEURL = "https://api-dev.platform.wellytics.health/auth/api"
    ADMINID = "1caa1665-ff18-4247-9560-cf3ef2713dec"
elif ENV in ['STAGE', 'DEMO']:
    BASEURL = "https://api-stage.platform.wellytics.health/auth/api"
    ADMINID = "b97a2b09-65e9-4498-a8c9-d5a1d67bd314"
elif ENV == 'PROD':
    BASEURL = "https://api.platform.wellytics.health/auth/api"
    ADMINID = "beeb3418-4d76-47e2-984d-5a075218cb27"
else:
    raise ValueError("Set correct ENV: DEV / STAGE / DEMO / PROD")

# ── authenticate ─────────────────────────────────────
response = requests.post(
    f"{BASEURL}/authenticate",
    json={"username": "superadmin@wellytics.health", "password": "superadmin@123"},
    headers={"Content-Type": "application/json"}
)
auth_resp = response.json()
if not auth_resp.get("success") or not auth_resp.get("result", {}).get("token"):
    raise RuntimeError(f"Authentication failed: {auth_resp.get('message', auth_resp)}")
AUTHTOKEN = auth_resp["result"]["token"]

# ── headers — Bearer token + userId (both required by API) ──
HEADERS = {
    "Authorization": f"Bearer {AUTHTOKEN}",
    "Content-Type":  "application/json",
    "userId":        ADMINID,
}


print(f"ENV     : {ENV}")
print(f"BASEURL : {BASEURL}")
print(f"ORG_ID  : {ORG_ID}")
print(f"Token   : {AUTHTOKEN[:20]}...")
print("Authentication ✅")


In [5]:
# ──────────────────────────────────────────────────────
# CELL 3 — HELPERS
# ──────────────────────────────────────────────────────

# ── date range ────────────────────────────────────────
def _parse_date(s):
    s = s.strip()
    if not s: return None
    for fmt in ("%d/%m/%y", "%d/%m/%Y", "%Y-%m-%d"):
        try:
            return datetime.strptime(s, fmt).replace(tzinfo=timezone.utc)
        except ValueError:
            pass
    raise ValueError(f"Cannot parse date '{s}'. Use DD/MM/YY or DD/MM/YYYY")

dt_from = _parse_date(DATE_FROM)
dt_to   = _parse_date(DATE_TO)
if dt_to:
    dt_to = dt_to.replace(hour=23, minute=59, second=59)

def in_range(ts_str):
    if not ts_str or (dt_from is None and dt_to is None): return True
    try:
        ts = datetime.fromisoformat(ts_str.replace("Z", "+00:00"))
    except Exception:
        return True
    if dt_from and ts < dt_from: return False
    if dt_to   and ts > dt_to:   return False
    return True

def fmt_date(ts_str):
    if not ts_str: return ""
    try:
        return datetime.fromisoformat(ts_str.replace("Z", "+00:00")).strftime("%d %b %Y")
    except Exception:
        return ts_str[:10]

# ── paginated fetch ───────────────────────────────────
def get_all_pages(url, method="GET", payload=None, size=100):
    records, page = [], 1
    while True:
        sep   = "&" if "?" in url else "?"
        paged = f"{url}{sep}pgNum={page}&noOfRecords={size}"
        r = (requests.get(paged, headers=HEADERS)
             if method == "GET"
             else requests.post(paged, headers=HEADERS, json=payload or {}))
        if not r.ok:
            print(f"  ❌ {r.status_code} error for: {paged}")
            print(f"  Response: {r.text[:300]}")
            r.raise_for_status()
        data   = r.json()
        result = data.get("result", {})
        batch  = (result if isinstance(result, list)
                  else next((v for v in result.values() if isinstance(v, list)), []))
        if not batch: break
        records.extend(batch)
        total = (data.get("pagination") or {}).get("totalRecords")              or (result.get("totalRecords") if isinstance(result, dict) else None)
        if (total and len(records) >= int(total)) or len(batch) < size:
            break
        page += 1
    return records

# ── resolve OP / IP ───────────────────────────────────
def resolve_patient_type(pt):
    for key in ("appointmentType", "patientType", "visitType", "encounterType", "type"):
        val = pt.get(key)
        if val and str(val).strip():
            v = str(val).strip().upper()
            if v in ("OP", "OUTPATIENT", "OUT_PATIENT", "OUT-PATIENT"): return "OP"
            if v in ("IP", "INPATIENT", "IN_PATIENT", "IN-PATIENT"):   return "IP"
            return v
    return "—"

range_label = f"{DATE_FROM} → {DATE_TO}" if (DATE_FROM or DATE_TO) else "All time"
print(f"Date range : {range_label}")
print("Helpers ready ✅")


Date range : 25/03/26 → 25/03/26
Helpers ready ✅


In [6]:
# ──────────────────────────────────────────────────────
# CELL 4 — FETCH DATA
# ──────────────────────────────────────────────────────

# 1. Patients
print("Fetching patients...")
all_patients = get_all_pages(
    f"{BASEURL}/patients/summary?status=&organizationId={ORG_ID}"
)
patients = [p for p in all_patients if in_range(p.get("createdTs", ""))]
print(f"  Total fetched : {len(all_patients)}")
print(f"  After filter  : {len(patients)}  ({range_label})")

# 2. Prompt keys — fetched live every run so any new prompt added on the platform is included
# Two fetches combined:
#   a) /users/promptKeys                          -> generic (null organizationId) + all visible keys
#   b) /users/promptKeys?organizationId={ORG_ID} -> org-specific keys for this org
# Deduplicated by id so no duplicates
print("Fetching prompt keys...")
_pk_seen = {}   # id -> prompt object

# a. All keys visible to admin
try:
    r1 = requests.get(f"{BASEURL}/users/promptKeys", headers=HEADERS)
    if r1.status_code == 200:
        for pk in r1.json().get("result", []):
            if pk.get("id"):
                _pk_seen[pk["id"]] = pk
except Exception:
    pass

# b. Org-specific keys
try:
    r2 = requests.get(f"{BASEURL}/users/promptKeys?organizationId={ORG_ID}", headers=HEADERS)
    if r2.status_code == 200:
        for pk in r2.json().get("result", []):
            if pk.get("id"):
                _pk_seen[pk["id"]] = pk
except Exception:
    pass

_pk_full   = list(_pk_seen.values())
prompt_map = {pk["id"]: pk["key"] for pk in _pk_full}
n_generic  = sum(1 for pk in _pk_full if not pk.get("organizationId"))
n_org      = sum(1 for pk in _pk_full if pk.get("organizationId") == ORG_ID)
n_other    = sum(1 for pk in _pk_full if pk.get("organizationId") and pk.get("organizationId") != ORG_ID)

# Filter to only generic + this org's keys — skip other orgs entirely
_pk_filtered = [
    pk for pk in _pk_full
    if not pk.get("organizationId") or pk.get("organizationId") == ORG_ID
]
print(f"  {len(_pk_full)} total keys ({n_generic} generic + {n_org} this org + {n_other} other orgs)")
print(f"  {len(_pk_filtered)} keys will be checked per patient (generic + this org only)")

# 3. Files (bulk POST — exclude Notes/TEMP_PATIENT bucket)
print("Fetching file uploads...")
patient_ids       = [p["id"] for p in patients]
all_files         = get_all_pages(
    f"{BASEURL}/files-status?",
    method="POST",
    payload={"patientIds": patient_ids},
    size=200,
)
# File types that are NOT user-uploaded clinical files:
# TEMP_PATIENT  = the admission PDF used to create the patient profile
# NOTE/NOTES    = text notes saved via the notes API (not file uploads)
# TRANSCRIPTION = audio transcription (counted separately for drug detection)
# Any type containing COPILOT/AI/SUMMARY = AI-generated content, not uploads
# File types to EXCLUDE from the uploaded-file count.
# TEMP_PATIENT = the admission PDF that creates the patient profile (not an upload)
# NOTE/NOTES   = text notes (separate from file uploads)
# TRANSCRIPTION = audio file tracked separately for drug detection
# File types that are NOT user-uploaded clinical files.
# Uses both exact match AND substring check for note-like types.
# CLINICAL_NOTES is explicitly protected — it IS a real uploaded document.
_FTYPE_EXACT_EXCLUDE = {
    "TEMP_PATIENT",   # admission PDF used to create patient profile
    "NOTE",           # text note (exact)
    "NOTES",          # text notes (exact)
    "PATIENT_NOTE",   # patient note
    "TRANSCRIPTION",  # audio file — tracked separately for drug detection
}

# Substrings that indicate a fileType is a note/AI-generated, NOT a real upload
# These are checked only if the type is NOT a known clinical file type
_FTYPE_NOTE_PATTERNS = ("NOTE", "COPILOT", "AI_", "TRANSCRIPT_NOTE")

# Known real clinical file types — always count these, never block
_FTYPE_ALWAYS_COUNT = {
    "CLINICAL_NOTES", "BLOOD_TEST", "IMAGING", "RADIOLOGY",
    "LAB_REPORT", "PATHOLOGY", "ECG", "ECHO", "MRI", "CT_SCAN",
    "XRAY", "ULTRASOUND", "DISCHARGE_SUMMARY", "PRESCRIPTION",
    "REFERRAL", "OPERATION_NOTE", "OT_NOTE", "PROCEDURE_NOTE",
}

def _is_real_file(ftype):
    if not ftype:
        return False
    if ftype in _FTYPE_ALWAYS_COUNT:
        return True                     # always a real clinical file
    if ftype in _FTYPE_EXACT_EXCLUDE:
        return False                    # definitely not a file
    # Check note-like substrings only for unknown types
    for pattern in _FTYPE_NOTE_PATTERNS:
        if pattern in ftype:
            return False
    return True                         # unknown type — count it

files_by_patient:        dict = {}
transcription_patient_ids = set()

# Print all fileTypes found so you can verify what's being counted
_all_ftypes = sorted({str(f.get("fileType") or "").upper() for f in all_files})
_counted    = [ft for ft in _all_ftypes if _is_real_file(ft)]
_excluded   = [ft for ft in _all_ftypes if not _is_real_file(ft)]
print(f"  File types found   : {_all_ftypes}")
print(f"  Counted as uploads : {_counted}")
print(f"  Excluded           : {_excluded}")

for f in all_files:
    pid    = f.get("patientId")
    ftype  = str(f.get("fileType") or "").upper()
    if not pid:
        continue

    # Track transcription patients for drug detection (before exclusion)
    if ftype == "TRANSCRIPTION":
        transcription_patient_ids.add(pid)

    # Skip non-real-file types
    if not _is_real_file(ftype):
        continue

    # Skip failed uploads
    upload_status = str(f.get("uploadStatus") or "").upper()
    if upload_status == "FAILED":
        continue

    if in_range(f.get("uploadTime", "") or f.get("createdTs", "")):
        files_by_patient.setdefault(pid, []).append(f)

print(f"  {len(all_files)} file records (TEMP_PATIENT excluded from counts)")
print(f"  {len(transcription_patient_ids)} patients have transcription files")

# 4. Doctor names
# The patient detail endpoint GET /{patientId}/patients returns
# patient['doctors'][0] with the assigned doctor id + name.
# The userId on the patient/file object is the medical officer who created
# the record — NOT the assigned doctor.
print("Fetching doctor details per patient...")

# Build base URL without /auth/api suffix for the detail endpoint
_base_root = BASEURL.replace("/auth/api", "").rstrip("/")

doctor_by_patient: dict = {}   # patientId → doctor display name

# Collect all unique doctor IDs from the summary response first
# (some API versions return doctorId / doctorName directly on the summary object)
doctor_id_set = set()
for pt in patients:
    for field in ("doctorId", "assignedDoctorId"):
        did = pt.get(field)
        if did:
            doctor_id_set.add(did)

# Bulk-fetch all org users once for name resolution
user_name_map: dict = {}   # userId → display name
try:
    ur = requests.get(
        f"{BASEURL}/users?organizationId={ORG_ID}&pgNum=1&noOfRecords=500",
        headers=HEADERS
    )
    if ur.status_code == 200:
        if not ur.text or not ur.text.strip():
            udata = {}
        else:
            try:
                udata = ur.json().get("result", {})
            except Exception:
                udata = {}
        ulist = udata if isinstance(udata, list) else (
            udata.get("users") or udata.get("data") or []
        )
        for u in (ulist if isinstance(ulist, list) else []):
            uid  = u.get("id") or u.get("userId")
            name = (
                u.get("name")
                or f"{u.get('firstName','')} {u.get('lastName','')}".strip()
                or u.get("email", "")
            )
            if uid and name:
                user_name_map[uid] = name
        print(f"  Bulk user fetch: {len(user_name_map)} users")
except Exception as e:
    print(f"  Bulk user fetch failed: {e}")

# For each patient, get the assigned doctor from patient detail endpoint
for pt in patients:
    pid = pt.get("id", "")
    if not pid:
        continue

    # First try fields directly on the summary object
    doc_name = pt.get("doctorName") or pt.get("assignedDoctorName") or ""
    doc_id   = pt.get("doctorId")   or pt.get("assignedDoctorId")   or ""

    if not doc_name and doc_id:
        doc_name = user_name_map.get(doc_id, "")

    if not doc_name:
        # Fetch full patient detail: GET /{patientId}/patients
        try:
            url  = f"{_base_root}/auth/api/{pid}/patients"
            resp = requests.get(url, headers=HEADERS)
            if resp.status_code == 200:
                if not resp.text or not resp.text.strip():
                    result = {}
                else:
                    try:
                        result = resp.json().get("result", {})
                    except Exception:
                        result = {}
                if isinstance(result, list):
                    patient_obj = result[0] if result else {}
                elif isinstance(result, dict):
                    patient_obj = result.get("patients", [result])[0] if result.get("patients") else result
                else:
                    patient_obj = {}

                # doctors array — assigned doctor(s)
                doctors = patient_obj.get("doctors", [])
                if doctors:
                    d = doctors[0]
                    doc_name = (
                        d.get("name")
                        or f"{d.get('firstName','')} {d.get('lastName','')}".strip()
                        or d.get("email", "")
                        or d.get("id", "")
                    )
        except Exception:
            pass

    doctor_by_patient[pid] = doc_name or "—"

print(f"  Doctor resolved for {sum(1 for v in doctor_by_patient.values() if v != '—')} / {len(patients)} patients")
print("Fetch complete ✅")


Fetching patients...
  Total fetched : 538
  After filter  : 5  (25/03/26 → 25/03/26)
Fetching prompt keys...
  91 total keys (4 generic + 19 this org + 68 other orgs)
  23 keys will be checked per patient (generic + this org only)
Fetching file uploads...
  File types found   : ['', 'AUDIO_CHUNKS', 'BIOPSY', 'BLOOD_TEST', 'CLINICAL_DATA', 'CLINICAL_NOTES', 'CT_SCAN', 'DOCTOR_KYC', 'ECG', 'GENETIC_ORDER_REPORTS', 'GENETIC_ORDER_RESULTS', 'MRI', 'ORGANIZATION_LOGO', 'TEMP_PATIENT', 'TRANSCRIPTION', 'ULTRASOUND', 'URINE_TEST']
  Counted as uploads : ['AUDIO_CHUNKS', 'BIOPSY', 'BLOOD_TEST', 'CLINICAL_DATA', 'CLINICAL_NOTES', 'CT_SCAN', 'DOCTOR_KYC', 'ECG', 'GENETIC_ORDER_REPORTS', 'GENETIC_ORDER_RESULTS', 'MRI', 'ORGANIZATION_LOGO', 'ULTRASOUND', 'URINE_TEST']
  Excluded           : ['', 'TEMP_PATIENT', 'TRANSCRIPTION']
  17225 file records (TEMP_PATIENT excluded from counts)
  348 patients have transcription files
Fetching doctor details per patient...
  Bulk user fetch: 500 users
  Doct

In [7]:
from openai import OpenAI
import os, base64
from dotenv import load_dotenv

load_dotenv()

os.environ["OPENAI_API_KEY"] = "xyz"

# Your username and password
username = 'rep-user'
password = 'De8936129bWqe_ko'

# Create the credentials string
credentials = f"{username}:{password}"

# Encode in Base64
encoded_credentials = base64.b64encode(credentials.encode('utf-8')).decode('utf-8')

# Headers with the Basic Authentication
headers = {
    "Authorization": f"Basic {encoded_credentials}",
    "Content-Type": "application/json"
}

if ENV == 'DEV':
     base_url = "https://ada-discovery-dev.wellytics.health/api2"
elif ENV in ['STAGE', 'DEMO']:
     base_url = "https://ada-discovery-stage.wellytics.health/api2"
elif ENV == 'PROD':
     base_url = "https://ada-discovery.wellytics.health/api2"

print("Using ENV:", ENV)
print("Base URL:", base_url)

Using ENV: PROD
Base URL: https://ada-discovery.wellytics.health/api2


In [8]:
prompts_available = [pk for pk in _pk_full if pk["generic"] or pk.get("organizationId") == ORG_ID]
prompts_dict = {pk["description"]: pk["key"] for pk in prompts_available}

bot_ids = ["patient:"+patient for patient in patient_ids]

api_endpoint = f"{base_url}/listConversationThreadsStreamed"

payload = {
  "botIds": bot_ids,
  "includeSources": False,
  "includeUserPrompt": False,
  "scrollBatchSize": 2
}

response = requests.post(api_endpoint, headers=headers, json=payload, stream=True, timeout=60)
data = response.json()

prompts_used_dict = {}

for d in data:
    bot_id = d["botId"]
    bot_id = bot_id.split(":")[1]
    for entry in d["entries"]:
        query = entry["query"]
        if query in prompts_dict:
            query = prompts_dict[query]
        if query != "Detailed summary" and query != "Timeline":
            if bot_id not in prompts_used_dict:
                prompts_used_dict[bot_id] = query
            else:
                prompts_used_dict[bot_id] += ", " + query

prompts_used_dict

{'4321cfa0-9b6d-48cf-8349-63cd2f27e6f8': 'ENT Discharge Summary',
 '4993b9b8-da33-45cf-83c5-5530cf5faf42': 'Neurology Discharge Summary',
 '8e102442-6fb7-4af4-bc6e-12ca7840fe77': 'ENT Discharge Summary',
 'd03e551e-80ca-4b64-9b7b-d60bac8dced6': 'ENT Discharge Summary',
 'f92c4723-c5fa-4496-a08f-18e1bb16ac57': 'ENT Discharge Summary'}

In [10]:
import datetime
import time

def parse_date_any_format(date_str):
    """Parse date in either %d/%m/%Y or %d/%m/%y formats."""
    for fmt in ("%d/%m/%Y", "%d/%m/%y"):
        try:
            return datetime.datetime.strptime(date_str, fmt)
        except ValueError:
            continue
    raise ValueError(f"Date '{date_str}' is not in an accepted format.")

def ist_epoch_millis(dt):
    """Convert naive dt to milliseconds since epoch in IST (+05:30 offset)."""
    # Add 5 hours 30 minutes for IST offset
    ist_offset = datetime.timedelta(hours=5, minutes=30)
    dt_ist = dt + ist_offset
    # Timezone-naive epoch calc (local time as IST)
    epoch = datetime.datetime(1970, 1, 1)
    return int((dt_ist - epoch).total_seconds() * 1000)

# Parse to datetime objects (midnight as default time)
if DATE_FROM:
    from_dt = parse_date_any_format(DATE_FROM)
    From = ist_epoch_millis(from_dt)
if DATE_TO:
    to_dt = parse_date_any_format(DATE_TO)
    To = ist_epoch_millis(to_dt)

api_endpoint = f"{base_url}/listTranscriptionResults"

payload = {
    "from": From,
    "to": To
}

response = requests.get(api_endpoint, headers=headers, params=payload)
data = response.json()
data = [d for d in data if d["result"]["organizationId"] == ORG_ID]
for d in data:
    d["patientId"] = d["s3FileKey"].split("/")[0]

transcription_dict = {}
for d in data:
    pid = d["patientId"]
    if pid not in transcription_dict:
        transcription_dict[pid] = {}
    if 'transcriptBeforeCorrection' in d["result"]:
        if "On" in transcription_dict[pid]:
            transcription_dict[pid]["On"] += 1 
        else:
            transcription_dict[pid]["On"] = 1
    else:
        if "Off" in transcription_dict[pid]:
            transcription_dict[pid]["Off"] += 1 
        else:
            transcription_dict[pid]["Off"] = 1
transcription_dict

{}

In [ ]:
# ──────────────────────────────────────────────────────────────────────
# CELL 5 — PATIENT-WISE DETAILS (OFFLINE NOTE MATCHING)
# ──────────────────────────────────────────────────────────────────────
import re
import time as _time

def _fetch_notes_with_retry(pid, max_retries=3, delay=2):
    """Fetch notes with retry on 404 or error."""
    for attempt in range(max_retries):
        try:
            nr = requests.get(f"{BASEURL}/patients/{pid}/notes", headers=HEADERS, timeout=15)
            if nr.status_code == 200:
                if not nr.text or not nr.text.strip():
                    return []
                try:
                    result = nr.json().get("result", [])
                    return result if isinstance(result, list) else []
                except Exception:
                    return []
            elif nr.status_code == 404:
                if attempt < max_retries - 1:
                    _time.sleep(delay)
                    continue
                return []
            else:
                return []
        except Exception:
            if attempt < max_retries - 1:
                _time.sleep(delay)
            continue
    return []

def _guess_prompts_from_notes(copilot_notes):
    """
    Reverse-engineers which prompt was used by matching headings
    in the generated note against the prompt descriptions.
    """
    matched_labels = set()

    for note in copilot_notes:
        text = str(note.get("notes") or "")
        text_lower = text.lower()
        if not text_lower:
            continue

        # Extract markdown headings from the generated note
        note_headings = [h.strip() for h in re.findall(r'###\s*([^\n:]+)', text_lower) if h.strip()]

        best_match = ""
        max_score = 0

        for pk in _pk_full:
            desc = str(pk.get("description") or "").lower()
            key  = str(pk.get("key") or "").strip()
            if not desc or not key:
                continue

            score = 0

            # Score based on heading overlap (+2 per matched heading)
            for heading in note_headings:
                if heading in desc:
                    score += 2

            # Score based on key name appearing in note (+3)
            if len(key) > 3 and key.lower() in text_lower:
                score += 3

            if score > max_score:
                max_score = score
                best_match = key

        if best_match and max_score > 0:
            matched_labels.add(best_match)

    return matched_labels


# ── Main loop ───────────────────────────────────────────────────────
rows  = []
total = len(patients)

for i, pt in enumerate(patients, 1):
    pid          = pt.get("id", "")
    created_date = fmt_date(pt.get("createdTs", ""))
    doctor       = doctor_by_patient.get(pid, "")
    patient_name = (
        pt.get("name")
        or f"{pt.get('firstName','')} {pt.get('lastName','')}".strip()
        or pid[:8]
    )
    patient_type = resolve_patient_type(pt)

    # Files Uploaded
    num_files = sum(1 for f in files_by_patient.get(pid, []))

    # Drug Detection
    drug_detection = transcription_dict[pid] if pid in transcription_dict else ""

    # Fetch notes with retry
    all_notes = _fetch_notes_with_retry(pid)
    notes_in_range = all_notes
    if DATE_FROM or DATE_TO:
        notes_in_range = [
            n for n in all_notes
            if in_range(n.get("createdTs", "") or n.get("updatedTs", ""))
        ]

    # Note Saved
    notes_saved = "Yes" if notes_in_range else "No"

    # CoPilot notes — auto-created whenever a prompt is run
    copilot_notes = [
        n for n in notes_in_range
        if "CoPilotResponse" in str(n.get("tags", ""))
    ]

    # Prompt Used — offline matching, zero API calls
    # matched_labels = _guess_prompts_from_notes(copilot_notes)
    # prompt_used = ", ".join(sorted(matched_labels)) if matched_labels else ""
    prompt_used = prompts_used_dict[pid] if pid in prompts_used_dict else ""

    rows.append({
        "Date"           : created_date,
        "Doctor"         : doctor,
        "Patient"        : patient_name,
        "OP / IP"        : patient_type,
        "Files Uploaded" : num_files,
        "Drug Detection" : drug_detection,
        "Prompt Used"    : prompt_used,
        "Note Saved"     : notes_saved,
    })

    print(f"  [{i}/{total}] {patient_name}", end="\r")

print(f"\nDone — {len(rows)} rows built")
df = pd.DataFrame(rows)
df["_sort_date"] = pd.to_datetime(df["Date"], format="%d %b %Y", errors="coerce")
df["_row_order"]  = range(len(df))
df = (df.sort_values(["_sort_date", "_row_order"], ascending=[True, True])
        .drop(columns=["_sort_date", "_row_order"])
        .reset_index(drop=True))
df = df[["Date", "Doctor", "Patient", "OP / IP", "Files Uploaded", "Drug Detection", "Prompt Used", "Note Saved"]]
print("\n" + "=" * 70)
print(" PATIENT-WISE DETAILS")
print("=" * 70)
pd.set_option("display.max_rows", None)
pd.set_option("display.max_colwidth", None)
pd.set_option("display.width", 1000)
display(df)


  [5/5] Naga Munishwara Rao Yarragadla
Done — 5 rows built

 PATIENT-WISE DETAILS


,Date,Doctor,Patient,OP / IP,Files Uploaded,Drug Detection,Prompt Used,Note Saved
0,2026-03-25,Brn Padmini,varala nagamani,OP,1,,ENT Discharge Summary,No
1,2026-03-25,Mohammad Mahaboob Shareef,C PHANI MADHAV,IP,1,,ENT Discharge Summary,No
2,2026-03-25,Jonnalagadda Dheeraj Kumar,Shaik nagur vali,IP,2,,ENT Discharge Summary,No
3,2026-03-25,Jonnalagadda Dheeraj Kumar,sowmya m,IP,1,,ENT Discharge Summary,Yes
4,2026-03-25,Santhosh Kumar B,Naga Munishwara Rao Yarragadla,IP,4,,Neurology Discharge Summary,No


In [12]:
# ─────────────────────────────────────────────────────────────
# CELL 6 — OVERALL SUMMARY
# ─────────────────────────────────────────────────────────────
total_patients = len(df)
total_op       = (df['OP / IP'] == 'OP').sum()
total_ip       = (df['OP / IP'] == 'IP').sum()
total_files    = df['Files Uploaded'].sum()
drug_on_count  = (df['Drug Detection'] == 'On').sum()
drug_off_count = (df['Drug Detection'] == 'Off').sum()
notes_yes      = (df['Note Saved'].str.startswith('Yes')).sum()
notes_no       = (df['Note Saved'] == 'No').sum()

prompts_series   = df[df['Prompt Used'] != '--']['Prompt Used'].str.split(', ')
all_prompts_flat = [p for sub in prompts_series for p in sub] if len(prompts_series) else []
prompt_counts    = Counter(all_prompts_flat)
all_prompts_str  = ', '.join(f'{k} ({v})' for k, v in prompt_counts.most_common()) or '--'

doctor_summary = (
    df.groupby('Doctor')
      .agg(
          Patients       = ('Patient',        'count'),
          OP             = ('OP / IP',         lambda x: (x == 'OP').sum()),
          IP             = ('OP / IP',         lambda x: (x == 'IP').sum()),
          Files_Uploaded = ('Files Uploaded',  'sum'),
          Notes_Saved    = ('Note Saved',      lambda x: x.str.startswith('Yes').sum()),
          Prompts_Used   = ('Prompt Used',     lambda x: (x != '--').sum()),
      )
      .reset_index()
      .rename(columns={'Files_Uploaded': 'Files Uploaded',
                       'Notes_Saved':    'Notes Saved',
                       'Prompts_Used':   'Prompts Used'})
      .sort_values('Patients', ascending=False)
      .reset_index(drop=True)
)

print('\n' + '=' * 70)
print(' OVERALL SUMMARY')
print('=' * 70)
summary_kv = [
    ('Period',                    range_label),
    ('Environment',               ENV),
    ('Organisation ID',           ORG_ID),
    ('',                          ''),
    ('Total Patients',            total_patients),
    (' OP (Outpatient)',           int(total_op)),
    (' IP (Inpatient)',            int(total_ip)),
    (' Type Unknown',              int(total_patients - total_op - total_ip)),
    ('',                          ''),
    ('Total Files Uploaded',      int(total_files)),
    ('',                          ''),
    ('Patients w/ Transcription', int(drug_on_count + drug_off_count)),
    (' Drug Detection On',        int(drug_on_count)),
    (' Drug Detection Off',       int(drug_off_count)),
    ('',                          ''),
    ('Notes Saved',               f'{int(notes_yes)} (not saved: {int(notes_no)})'),
    ('Patients w/ Prompt Used',   int((df['Prompt Used'] != '--').sum())),
    ('Prompts Used',              all_prompts_str),
]
summary_df = pd.DataFrame(summary_kv, columns=['Metric', 'Value'])
print(summary_df.to_string(index=False))
print('\n' + '-' * 70)
print(' SUMMARY BY DOCTOR')
print('-' * 70)
display(doctor_summary)



 OVERALL SUMMARY
                   Metric                                                      Value
                   Period                                        25/03/26 → 25/03/26
              Environment                                                       PROD
          Organisation ID                       360dd06e-ce7f-4d93-ba44-d9d63189f1d5
                                                                                    
           Total Patients                                                          5
          OP (Outpatient)                                                          1
           IP (Inpatient)                                                          4
             Type Unknown                                                          0
                                                                                    
     Total Files Uploaded                                                          9
                                               

,Doctor,Patients,OP,IP,Files Uploaded,Notes Saved,Prompts Used
0,Jonnalagadda Dheeraj Kumar,2,0,2,3,1,2
1,Brn Padmini,1,1,0,1,0,1
2,Mohammad Mahaboob Shareef,1,0,1,1,0,1
3,Santhosh Kumar B,1,0,1,4,0,1


In [13]:
# ──────────────────────────────────────────────────────
# CELL 7 — EXPORT TO EXCEL  (3 sheets)
# ──────────────────────────────────────────────────────

date_tag  = (
    f"{DATE_FROM.replace('/','')}-{DATE_TO.replace('/','')}"
    if (DATE_FROM or DATE_TO) else "all"
)
xlsx_file = f"analytics_{ENV}_{ORG_ID[:8]}_{date_tag}.xlsx"

with pd.ExcelWriter(xlsx_file, engine="openpyxl") as writer:
    df.to_excel(writer,            sheet_name="Patient-wise Details", index=False)
    doctor_summary.to_excel(writer, sheet_name="Summary by Doctor",    index=False)
    summary_df.to_excel(writer,    sheet_name="Overall Summary",       index=False)

print(f"Excel saved → {xlsx_file}  (3 sheets: Patient-wise Details | Summary by Doctor | Overall Summary)")

Excel saved → analytics_PROD_360dd06e_250326-250326.xlsx  (3 sheets: Patient-wise Details | Summary by Doctor | Overall Summary)
